# 01 - Paramétrage et lancement du contrôle RGPD Article 9

Notebook de pilotage local : référentiels, scoring, lemmatisation spaCy, familles lexicales et lancement du batch.

In [9]:
from pathlib import Path
import sys
import pandas as pd
import yaml

PROJECT_ROOT = Path("..").resolve()
SRC_DIR = PROJECT_ROOT / "src"
sys.path.append(str(SRC_DIR))

PDF_DIR = PROJECT_ROOT / "input" / "pdf"
REFERENTIEL_DIR = PROJECT_ROOT / "input" / "referentiel"
CONFIG_FILE = PROJECT_ROOT / "config" / "scoring.yaml"
OUTPUT_CSV = PROJECT_ROOT / "output" / "resultats_controle.csv"
OUTPUT_XLSX = PROJECT_ROOT / "output" / "resultats_controle.xlsx"

print("Projet :", PROJECT_ROOT)

Projet : E:\workspace\PocNLPGpt


## 1. Vérification des dépendances locales

In [10]:
# Si spaCy ou le modèle français ne sont pas installés, exécuter dans un terminal :
# pip install -r ../requirements.txt
# python -m spacy download fr_core_news_md

import spacy
try:
    nlp = spacy.load("fr_core_news_md", disable=["ner"])
    print("Modèle spaCy français OK : fr_core_news_md")
except OSError as exc:
    raise RuntimeError("Modèle spaCy manquant. Exécuter : python -m spacy download fr_core_news_md") from exc

Modèle spaCy français OK : fr_core_news_md


## 2. Vérifier les PDF et référentiels

In [ ]:
pdf_files = sorted(PDF_DIR.glob("*.pdf"))
print(f"Nombre de PDF trouvés : {len(pdf_files)}")
for pdf in pdf_files[:20]:
    print("-", pdf.name)

required_files = [
    REFERENTIEL_DIR / "mots_interdits.csv",
    REFERENTIEL_DIR / "synonymes.csv",
    REFERENTIEL_DIR / "familles_lexicales.csv",
    REFERENTIEL_DIR / "domaines_semantiques.csv",
    REFERENTIEL_DIR / "whitelist.csv",
    CONFIG_FILE,
]

for file in required_files:
    print(file.name, "OK" if file.exists() else "MANQUANT")

Nombre de PDF trouvés : 8
- adhesion_emilie_bernard_conseil_non_professionnel.pdf
- adhesion_jean_dupont_diabete.pdf
- adhesion_julien_martin.pdf
- adhesion_laurent_moreau_capacite_inadaptee.pdf
- adhesion_nadia_benali.pdf
- adhesion_patrice_leroy_clause_beneficiaire_imprecise.pdf
- adhesion_sandrine_roche_formule_par_defaut_fautes.pdf
- souscriptions_assurance_vie_remplies.pdf
mots_interdits.csv OK
synonymes.csv OK
familles_lexicales.csv OK
whitelist.csv OK
scoring.yaml OK


## 3. Charger les référentiels métier

In [ ]:
mots_interdits = pd.read_csv(REFERENTIEL_DIR / "mots_interdits.csv", encoding="utf-8")
synonymes = pd.read_csv(REFERENTIEL_DIR / "synonymes.csv", encoding="utf-8")
familles_lexicales = pd.read_csv(REFERENTIEL_DIR / "familles_lexicales.csv", encoding="utf-8")
domaines_semantiques = pd.read_csv(REFERENTIEL_DIR / "domaines_semantiques.csv", encoding="utf-8")
whitelist = pd.read_csv(REFERENTIEL_DIR / "whitelist.csv", encoding="utf-8")

display(mots_interdits.head(20))
display(synonymes.head(20))
display(familles_lexicales.head(20))
display(domaines_semantiques.head(20))
display(whitelist.head(20))

,terme,categorie,gravite,commentaire
0,cancer,sante,100,Donnée de santé potentielle
1,maladie,sante,100,Donnée de santé potentielle
2,handicap,sante,100,Donnée de santé potentielle
3,hospitalisation,sante,100,Donnée de santé potentielle
4,traitement médical,sante,100,Donnée de santé potentielle
5,syndicat,appartenance_syndicale,100,Appartenance syndicale potentielle
6,musulman,religion,100,Conviction religieuse potentielle
7,catholique,religion,100,Conviction religieuse potentielle
8,juif,religion,100,Conviction religieuse potentielle
9,orientation sexuelle,vie_sexuelle,100,Donnée vie sexuelle potentielle


,terme_reference,synonyme,categorie,gravite
0,maladie,pathologie,sante,80
1,maladie,affection,sante,80
2,maladie,trouble de santé,sante,80
3,cancer,tumeur,sante,90
4,handicap,invalidité,sante,85
5,hospitalisation,séjour hospitalier,sante,85
6,traitement médical,soins médicaux,sante,85
7,religion,croyance religieuse,religion,80
8,syndicat,organisation syndicale,appartenance_syndicale,80


,terme_reference,variante,categorie,gravite
0,maladie,malade,sante,85
1,maladie,malades,sante,85
2,maladie,maladif,sante,80
3,maladie,pathologique,sante,80
4,cancer,cancéreux,sante,90
5,cancer,cancéreuse,sante,90
6,handicap,handicapé,sante,90
7,handicap,handicapée,sante,90
8,handicap,invalide,sante,85
9,hospitalisation,hospitalisé,sante,85


,expression,categorie,commentaire
0,assurance maladie,sante,Expression administrative fréquente
1,code de la santé publique,sante,Référence juridique ou réglementaire
2,questionnaire médical,sante,Expression métier autorisée selon contexte
3,contrat collectif santé,sante,Produit ou contexte administratif


## 4. Paramétrer le scoring

In [ ]:
with open(CONFIG_FILE, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

# Paramètres modifiables pour ce lancement
config["detection"]["activer_lemmatisation"] = True
config["detection"]["activer_familles_lexicales"] = True
config["detection"]["activer_synonymes"] = True
config["detection"]["activer_mots_proches"] = True
config["detection"]["activer_domaines_semantiques"] = True
config["detection"]["enrichir_avec_camembert"] = False  # Active CamemBERT pour enrichissement automatique
config["detection"]["domaines_a_analyser"] = ["sante", "religion"]
config["detection"]["seuil_similarite_orthographique"] = 88
config["detection"]["seuil_similarite_semantique"] = 0.6
config["detection"]["taille_contexte_mots"] = 12

config

{'score': {'mot_interdit_exact': 100,
  'lemme': 85,
  'famille_lexicale': 85,
  'synonyme': 80,
  'mot_proche': 60,
  'occurrence_multiple': 10},
 'seuils': {'risque_faible': 30,
  'a_surveiller': 60,
  'risque_fort': 80,
  'non_conforme': 100},
 'detection': {'activer_lemmatisation': True,
  'activer_familles_lexicales': True,
  'activer_synonymes': True,
  'activer_mots_proches': True,
  'seuil_similarite_orthographique': 88,
  'taille_contexte_mots': 12,
  'ignorer_mots_trop_courts': 4,
  'dedoublonner_alertes': True},
 'sortie': {'inclure_contexte': True,
  'format_csv': True,
  'format_excel': True}}

## 5. Test rapide de lemmatisation

In [14]:
from lemmatizer import lemmatize_text

texte_test = "Le client indique être malade et suivre un traitement."
tokens = lemmatize_text(texte_test)
pd.DataFrame(tokens)

,text,lemma,pos,start,end
0,le,le,DET,0,2
1,client,client,NOUN,3,9
2,indique,indique,VERB,10,17
3,etre,etre,AUX,18,22
4,malade,malade,ADJ,23,29
5,et,et,CCONJ,30,32
6,suivre,suivre,VERB,33,39
7,un,un,DET,40,42
8,traitement,traitement,NOUN,43,53


## 6. Lancer le traitement

In [15]:
from extract_pdf import extract_text_from_pdf
from load_referential import load_referentials
from detection import analyze_document
from export_results import export_results

referentials = load_referentials(REFERENTIEL_DIR)
results = []

for pdf_path in sorted(PDF_DIR.glob("*.pdf")):
    print("Analyse :", pdf_path.name)
    text_by_page = extract_text_from_pdf(pdf_path)
    results.extend(analyze_document(pdf_path, text_by_page, referentials, config))

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
export_results(results, OUTPUT_CSV, OUTPUT_XLSX)
print("CSV généré :", OUTPUT_CSV)
print("Excel généré :", OUTPUT_XLSX)

Analyse : adhesion_emilie_bernard_conseil_non_professionnel.pdf
Analyse : adhesion_jean_dupont_diabete.pdf
Analyse : adhesion_julien_martin.pdf
Analyse : adhesion_laurent_moreau_capacite_inadaptee.pdf
Analyse : adhesion_nadia_benali.pdf
Analyse : adhesion_patrice_leroy_clause_beneficiaire_imprecise.pdf
Analyse : adhesion_sandrine_roche_formule_par_defaut_fautes.pdf
Analyse : souscriptions_assurance_vie_remplies.pdf
CSV généré : E:\workspace\PocNLPGpt\output\resultats_controle.csv
Excel généré : E:\workspace\PocNLPGpt\output\resultats_controle.xlsx


## 7. Aperçu des résultats

In [16]:
df_results = pd.read_csv(OUTPUT_CSV, encoding="utf-8-sig")
display(df_results.head(50))
print("Documents :", df_results["document"].nunique())
print("Alertes :", len(df_results[df_results["score"] > 0]))

,document,score,score_unitaire,statut,terme_detecte,terme_reference,type_detection,categorie,motif,page,contexte
0,adhesion_emilie_bernard_conseil_non_profession...,75,75.0,À contrôler,traitement,traitement médical,famille lexicale,sante,Variante métier du terme interdit : traitement...,2.0,recue x exactitude des informations certifiee ...
1,adhesion_jean_dupont_diabete.pdf,0,NaN,Conforme apparent,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,adhesion_julien_martin.pdf,75,75.0,À contrôler,traitement,traitement médical,famille lexicale,sante,Variante métier du terme interdit : traitement...,2.0,recue x exactitude des informations certifiee ...
3,adhesion_laurent_moreau_capacite_inadaptee.pdf,75,75.0,À contrôler,traitement,traitement médical,famille lexicale,sante,Variante métier du terme interdit : traitement...,2.0,recue x exactitude des informations certifiee ...
4,adhesion_nadia_benali.pdf,75,75.0,À contrôler,traitement,traitement médical,famille lexicale,sante,Variante métier du terme interdit : traitement...,2.0,recue x exactitude des informations certifiee ...
5,adhesion_patrice_leroy_clause_beneficiaire_imp...,75,75.0,À contrôler,traitement,traitement médical,famille lexicale,sante,Variante métier du terme interdit : traitement...,2.0,recue x exactitude des informations certifiee ...
6,adhesion_sandrine_roche_formule_par_defaut_fau...,75,75.0,À contrôler,traitement,traitement médical,famille lexicale,sante,Variante métier du terme interdit : traitement...,2.0,recue x exactitude des informations certifiee ...
7,souscriptions_assurance_vie_remplies.pdf,100,75.0,Non conforme probable,traitement,traitement médical,famille lexicale,sante,Variante métier du terme interdit : traitement...,2.0,recue x exactitude des informations certifiee ...
8,souscriptions_assurance_vie_remplies.pdf,100,75.0,Non conforme probable,traitement,traitement médical,famille lexicale,sante,Variante métier du terme interdit : traitement...,5.0,recue x exactitude des informations certifiee ...


Documents : 8
Alertes : 8
